In [48]:
import torch
from datasets import load_dataset
import torch.nn as nn
from torch.utils.data import DataLoader
from sklearn.metrics import f1_score, roc_auc_score, accuracy_score
import torchvision
from torchvision import models
import torchvision.transforms as tr
from torch.cuda.amp import autocast
from sklearn.model_selection import train_test_split
import torchvision.io
import os
import time
!pip install mlflow
import mlflow
import numpy as np

In [2]:
device = ("cuda" if torch.cuda.is_available() else "cpu")
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))
print(torch.cuda.memory_allocated() / 1024**3)
print(device)

True
Tesla T4
0.0
cuda


In [3]:
dataset = load_dataset("ethz/food101")

README.md:   0%|          | 0.00/16.4k [00:00<?, ?B/s]

data/train-00000-of-00008.parquet: reconstructing file:   0%|          |  0.00B /  490MB            

data/train-00000-of-00008.parquet: downloading bytes:           |  0.00B            

data/train-00001-of-00008.parquet: reconstructing file:   0%|          |  0.00B /  464MB            

data/train-00001-of-00008.parquet: downloading bytes:           |  0.00B            

...

data/validation-00001-of-00003.parquet: downloading bytes:           |  0.00B            

data/validation-00002-of-00003.parquet: reconstructing file:   0%|          |  0.00B /  426MB            

data/validation-00002-of-00003.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/75750 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/25250 [00:00<?, ? examples/s]

In [21]:
test_val_splits = dataset['train'].train_test_split(test_size=0.2, seed=42)

In [35]:
train_dataset = test_val_splits['train']
validation_dataset = test_val_splits['test']
test_dataset = dataset['validation']
train_dataset = train_dataset.select(range(10000))
validation_dataset = validation_dataset.select(range(2000))
test_dataset = test_dataset.select(range(2000))

In [36]:
train_transform = tr.Compose([
    tr.Resize(256),
    tr.RandomResizedCrop(224),
    tr.RandomHorizontalFlip(0.5),
    tr.ToTensor(),
    tr.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])])

test_transform = tr.Compose([
    tr.Resize(256),
    tr.CenterCrop(224),
    tr.ToTensor(),
    tr.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])])

In [37]:
def train_transforms(examples):
    examples["pixel_values"] = [train_transform(image.convert("RGB")) for image in examples["image"]]
    return examples


def test_transforms(examples):
    examples["pixel_values"] = [test_transform(image.convert("RGB")) for image in examples["image"]]
    return examples


train_dataset.set_transform(train_transforms)
validation_dataset.set_transform(test_transforms)
test_dataset.set_transform(test_transforms)

In [38]:
sample = train_dataset[0]
print(sample.keys())


dict_keys(['image', 'label', 'pixel_values'])


In [39]:
def collate_fn(batch):
    pixel_values = torch.stack([
        item["pixel_values"]
        for item in batch
    ])

    labels = torch.tensor([
        item["label"]
        for item in batch
    ])

    return {
        "pixel_values": pixel_values,
        "label": labels
    }
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, collate_fn=collate_fn, num_workers=0, pin_memory=True)
validation_loader = DataLoader(validation_dataset, batch_size=64, shuffle=False, collate_fn=collate_fn, num_workers=0, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False, collate_fn=collate_fn)

In [40]:
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

for param in model.parameters():
  param.requires_grad = False

for param in model.layer4.parameters():
  param.requires_grad = True

model.fc = nn.Linear(512, 101)
model = model.to(device)

optimizer = torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr = 1e-4)
criterion = nn.CrossEntropyLoss()
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=2, factor=0.1)
scaler = torch.amp.GradScaler("cuda")

In [41]:
sample = train_dataset[0]

print(sample.keys())
print(sample["pixel_values"].shape)
print(sample["label"])


dict_keys(['image', 'label', 'pixel_values'])
torch.Size([3, 224, 224])
48


In [53]:
patience = 2
patience_counter = 0
best_val_score = float("inf")

mlflow.set_experiment("ResNet18 Food101")
mlflow.set_tracking_uri("https://attempt-evict-comment.ngrok-free.dev")

with mlflow.start_run(run_name="ResNet18_4th_layer_10_epoch"):
  mlflow.log_params({
    "model": "ResNet18",
    "pretrained": True,
    "batch_size": 64,
    "learning_rate": 1e-4,
    "epochs": 10,
    "max_samples": len(train_dataset),
    "optimizer": "Adam",
    "scheduler": "ReduceLROnPlateau",
    "frozen_layers": "all except layer4 and fc",
    "image_size": 224
})
  for epoch in range(10):
    print(f"\n===== STARTING EPOCH {epoch + 1}/10 =====", flush=True)
    start_epoch = time.time()
    model.train()
    train_start = time.time()
    train_loss = 0
    for batch in train_loader:
      optimizer.zero_grad()

      pixel_values = batch["pixel_values"].to(device, non_blocking=True)
      label = batch["label"].to(device, non_blocking=True)

      with torch.amp.autocast(device_type="cuda"):
        preds = model(pixel_values)
        loss = criterion(preds, label)

      scaler.scale(loss).backward()

      scaler.unscale_(optimizer)

      torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
      scaler.step(optimizer)
      scaler.update()

      train_loss += loss.item()

    train_time = time.time() - train_start
    print(
        f"Training finished for epoch {epoch + 1}",
        flush=True
    )
    model.eval()
    validation_start = time.time()
    all_preds = []
    all_labels = []
    all_probs = []
    test_loss = 0

    with torch.no_grad():
      for batch in validation_loader:
        pixel_values = batch["pixel_values"].to(device, non_blocking=True)
        label = batch["label"].to(device, non_blocking=True)

        with torch.amp.autocast(device_type='cuda'):
          preds = model(pixel_values)
          loss = criterion(preds, label)

        test_loss += loss.item()

        pred = torch.argmax(preds, dim=1)
        probs = torch.softmax(preds, dim=1)

        all_preds.extend(pred.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())
        all_labels.extend(label.cpu().numpy())

    validation_time = time.time() - validation_start

    print(f"Validation finished for epoch {epoch + 1}", flush=True)
    avg_test_loss = test_loss/len(validation_loader)
    avg_train_loss = train_loss/len(train_loader)

    accuracy = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average="weighted")
    scheduler.step(avg_test_loss)

    if avg_test_loss < best_val_score:
            best_val_score = avg_test_loss
            patience_counter = 0
            torch.save(model.state_dict(), "resnet18_food101_best.pt")
    else:
        patience_counter += 1


    print(f'Epoch {epoch + 1} train loss {avg_train_loss} test loss {avg_test_loss} accuracy {accuracy} f1 score {f1}')

    epoch_time = time.time() - start_epoch
    print(f"Train time: {train_time:.2f}s")
    print(f"Validation time: {validation_time:.2f}s")
    print(f"Epoch time: {epoch_time:.2f}s")

    mlflow.log_metrics({
      "train_loss": avg_train_loss,
      "val_loss": avg_test_loss,
      "accuracy": accuracy,
      "f1_score": f1
  }, step=epoch)
    if patience_counter == patience:
        print(f" Early stopping was activated on {epoch + 1}! End.")
        break


===== STARTING EPOCH 1/10 =====
Training finished for epoch 1
Validation finished for epoch 1
Epoch 1 train loss 0.6977457045369847 test loss 1.74128083512187 accuracy 0.57 f1 score 0.5688944272324113
Train time: 77.01s
Validation time: 12.59s
Epoch time: 89.70s

===== STARTING EPOCH 2/10 =====
Training finished for epoch 2
Validation finished for epoch 2
Epoch 2 train loss 0.6538044445833583 test loss 1.7229223288595676 accuracy 0.5825 f1 score 0.58061345735054
Train time: 76.96s
Validation time: 12.33s
Epoch time: 89.40s

===== STARTING EPOCH 3/10 =====
Training finished for epoch 3
Validation finished for epoch 3
Epoch 3 train loss 0.6072980283172267 test loss 1.7461013309657574 accuracy 0.5785 f1 score 0.5789446413960616
Train time: 75.64s
Validation time: 12.11s
Epoch time: 87.75s

===== STARTING EPOCH 4/10 =====
Training finished for epoch 4
Validation finished for epoch 4
Epoch 4 train loss 0.5489150988068551 test loss 1.7181979678571224 accuracy 0.5785 f1 score 0.5791233213475

In [ ]:
import mlflow

mlflow.set_tracking_uri(
    "https://attempt-evict-comment.ngrok-free.dev"
)

print(mlflow.get_tracking_uri())

mlflow.set_experiment("Test connection Colab")

with mlflow.start_run():
    mlflow.log_param("model", "ResNet18")
    mlflow.log_metric("accuracy", 0.85)